In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:  # aqui mantemos menor que o limite
            return True
        return False
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [4]:
analize = Analizer(0.8)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
1,model_3_4_17,0.798208,-2.032833,0.731565,-0.119054,0.656873,1.349381,20.280568,2.896889,1.726105,2.311497,1.592861,1.161629,0.627462,1.211081,21.400708,34.808342,"Hidden Size=[2, 1], regularizer=0.05, learning..."
3,model_3_4_16,0.797223,-2.010208,0.734131,-0.110550,0.659902,1.355971,20.129278,2.869193,1.712989,2.291091,1.618532,1.164462,0.625642,1.214035,21.390964,34.798598,"Hidden Size=[2, 1], regularizer=0.05, learning..."
4,model_3_4_15,0.796905,-2.395110,0.735891,-0.115283,0.660770,1.358098,22.703117,2.850198,1.720289,2.285244,1.617829,1.165374,0.625055,1.214987,21.387830,34.795464,"Hidden Size=[2, 1], regularizer=0.05, learning..."
6,model_3_4_14,0.794496,-1.837151,0.737935,-0.096285,0.664583,1.374206,18.972043,2.828139,1.690984,2.259562,1.658121,1.172265,0.620608,1.222171,21.364248,34.771882,"Hidden Size=[2, 1], regularizer=0.05, learning..."
7,model_14_0_9,0.793340,0.537512,-0.513735,-0.143621,-0.068755,1.381936,3.092658,2.104759,5.530488,3.817624,1.182380,1.175558,1.030428,1.225604,373.353029,601.282809,"Hidden Size=[10, 13], regularizer=0.2, learnin..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1432,model_11_0_13,0.077870,-0.035855,-0.076158,-0.554364,-0.439694,1.509996,1.696222,0.220996,2.206479,1.213738,1.507173,1.228819,23.131111,1.281132,49.175786,79.647682,"Hidden Size=[6], regularizer=0.05, learning_ra..."
1434,model_11_0_12,0.064384,-0.039008,-0.083499,-0.547766,-0.435033,1.532080,1.701385,0.222504,2.197113,1.209809,1.515979,1.237772,23.454788,1.290467,49.146747,79.618643,"Hidden Size=[6], regularizer=0.05, learning_ra..."
1437,model_11_0_11,0.051124,-0.041876,-0.089845,-0.537845,-0.427453,1.553793,1.706081,0.223807,2.183030,1.203418,1.523529,1.246512,23.773018,1.299579,49.118602,79.590498,"Hidden Size=[6], regularizer=0.05, learning_ra..."
1439,model_11_0_10,0.037910,-0.044608,-0.095995,-0.525793,-0.418056,1.575431,1.710555,0.225070,2.165922,1.195496,1.530747,1.255162,24.090160,1.308597,49.090942,79.562838,"Hidden Size=[6], regularizer=0.05, learning_ra..."
